# NegBench on COCO val2017 — does CLIP understand negation?

Two tasks from the released NegBench files, run with plain CLIP:

* **MCQ-Neg** — one image, four captions, pick the one that describes it. Chance = 25%.
* **Retrieval-Neg** — a negated caption as query, rank all 5000 images. Chance R@1 = 0.02%.

The headline number is not the overall accuracy but the **split by template**. A model that ignores
negation entirely still scores well on `positive` questions and collapses on `negative` ones; a model
that understands negation scores evenly. The gap between the two is the measurement.

**One caveat about the released file:** `correct_answer` is 0 on all 5914 rows — the right caption is
always `caption_0`. That does not bias CLIP, which scores each caption independently of its position,
but the options are shuffled below anyway (fixed seed) so the protocol stays valid for any evaluator.


## Config

In [ ]:
CONFIG = {
    "mcq_csv": "../../negbench/COCO/COCO_val_mcq_llama3.1_rephrased.csv",
    "retrieval_csv": "../../negbench/COCO/COCO_val_negated_retrieval_llama3.1_rephrased_affneg_true.csv",
    "image_dir": "../../negbench/COCO/val2017",   # csv paths are rewritten onto this folder

    "model_name": "openai/clip-vit-base-patch32",
    "device": None,             # None -> auto (mps / cuda / cpu)
    "image_batch_size": 64,
    "text_batch_size": 256,

    "shuffle_options": True,    # the released file always puts the answer at index 0
    "seed": 0,
    "subsample": None,          # e.g. 500 for a quick pass; None = full benchmark
    "recall_at": [1, 5, 10],
    "n_examples": 4,            # qualitative failures shown at the end
}

## Setup

In [ ]:
import ast
import hashlib
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, CLIPModel

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": False})
CACHE_DIR = Path("../data/cache"); CACHE_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR = Path(CONFIG["image_dir"]).expanduser()
rng = np.random.default_rng(CONFIG["seed"])


def pick_device(requested=None) -> str:
    if requested:
        return requested
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = pick_device(CONFIG["device"])
processor = AutoProcessor.from_pretrained(CONFIG["model_name"])
model = CLIPModel.from_pretrained(CONFIG["model_name"]).to(DEVICE).eval()
print(f"{CONFIG['model_name']} on {DEVICE}")


def _as_tensor(out):
    """transformers <5 returns a tensor from get_*_features; >=5 returns an output object."""
    if isinstance(out, torch.Tensor):
        return out
    for attr in ("image_embeds", "text_embeds", "pooler_output"):
        v = getattr(out, attr, None)
        if v is not None:
            return v
    return out[0]


def l2n(x):
    if isinstance(x, torch.Tensor):
        return x / x.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return x / np.clip(np.linalg.norm(x, axis=-1, keepdims=True), 1e-12, None)


def resolve(csv_path: str) -> Path:
    """The csv stores 'data/coco/images/val2017/xxx.jpg'; only the basename is reliable."""
    return IMAGE_DIR / Path(str(csv_path)).name


def _cached(key: str, fn):
    f = CACHE_DIR / f"negbench_{key}.npz"
    if f.exists():
        return np.load(f)["e"]
    E = fn()
    np.savez_compressed(f, e=E)
    return E


@torch.no_grad()
def embed_images(paths) -> np.ndarray:
    paths = list(paths)
    stamp = "\x00".join(f"{p.name}:{p.stat().st_size}:{p.stat().st_mtime_ns}" for p in paths)
    key = "img_" + hashlib.sha1((CONFIG["model_name"] + stamp).encode()).hexdigest()[:12]

    def run():
        bs, out, t0 = CONFIG["image_batch_size"], [], time.time()
        for s in range(0, len(paths), bs):
            imgs = [Image.open(p).convert("RGB") for p in paths[s:s + bs]]
            inp = processor(images=imgs, return_tensors="pt").to(DEVICE)
            out.append(l2n(_as_tensor(model.get_image_features(**inp))).float().cpu().numpy())
            d = min(s + bs, len(paths))
            print(f"\r  images {d}/{len(paths)} ({d / (time.time() - t0):.0f}/s)", end="", flush=True)
        print()
        return np.vstack(out).astype(np.float32)

    return _cached(key, run)


@torch.no_grad()
def embed_texts(texts) -> np.ndarray:
    texts = [str(t) for t in texts]
    key = "txt_" + hashlib.sha1((CONFIG["model_name"] + "\x00".join(texts)).encode()).hexdigest()[:12]

    def run():
        bs, out, t0 = CONFIG["text_batch_size"], [], time.time()
        for s in range(0, len(texts), bs):
            inp = processor(text=texts[s:s + bs], padding=True, truncation=True,
                            max_length=77, return_tensors="pt").to(DEVICE)
            out.append(l2n(_as_tensor(model.get_text_features(**inp))).float().cpu().numpy())
            d = min(s + bs, len(texts))
            print(f"\r  texts {d}/{len(texts)} ({d / (time.time() - t0):.0f}/s)", end="", flush=True)
        print()
        return np.vstack(out).astype(np.float32)

    return _cached(key, run)

## Caption typing

Only the *correct* caption's template is given in the file. To know **what CLIP picks when it is wrong**
the three distractors have to be typed too, which needs a rule. The rule is validated immediately below
against the ground-truth column: it is applied to the correct caption, where the true template is known,
and the agreement is reported. Do not trust the confusion matrix if that agreement is low.


In [ ]:
NEG_CUE = re.compile(
    r"\b(no|not|n't|without|absent|absence|lacks?|lacking|missing|neither|nor|none|"
    r"devoid|excluded|free of|nowhere|isn't|aren't|doesn't|don't)\b", re.I)
CONTRAST = re.compile(r"\b(but|while|however|whereas|although|though|yet)\b|;", re.I)


def caption_type(text: str) -> str:
    """positive = asserts only presence | negative = asserts only absence | hybrid = both."""
    t = str(text)
    if not NEG_CUE.search(t):
        return "positive"
    return "hybrid" if CONTRAST.search(t) else "negative"


CAP_COLS = [f"caption_{i}" for i in range(4)]
mcq = pd.read_csv(CONFIG["mcq_csv"])
if CONFIG["subsample"]:
    mcq = mcq.sample(n=min(CONFIG["subsample"], len(mcq)), random_state=CONFIG["seed"])
mcq = mcq.reset_index(drop=True)

# validate the rule where the truth is known: the correct caption
truth = mcq["correct_answer_template"].values
guess = np.array([caption_type(mcq.loc[i, CAP_COLS[mcq.loc[i, "correct_answer"]]])
                  for i in range(len(mcq))])
agree = float((truth == guess).mean())
print(f"caption typing rule vs ground truth on the correct caption: {agree:.1%} agreement "
      f"({len(mcq)} rows)")
print(pd.crosstab(pd.Series(truth, name="true"), pd.Series(guess, name="rule")).to_string())

## MCQ-Neg

In [ ]:
if CONFIG["shuffle_options"]:
    perm = np.array([rng.permutation(4) for _ in range(len(mcq))])
else:
    perm = np.tile(np.arange(4), (len(mcq), 1))

options = np.array([[mcq.loc[i, CAP_COLS[j]] for j in perm[i]] for i in range(len(mcq))], dtype=object)
answer = np.array([int(np.where(perm[i] == mcq.loc[i, "correct_answer"])[0][0]) for i in range(len(mcq))])
opt_type = np.vectorize(caption_type)(options)

paths = [resolve(p) for p in mcq["image_path"]]
uniq = sorted({p.name: p for p in paths}.values(), key=lambda p: p.name)
row_of = {p.name: i for i, p in enumerate(uniq)}
print(f"{len(mcq)} questions | {len(uniq)} unique images | {options.size} captions")

E_img_all = embed_images(uniq)
E_img = E_img_all[[row_of[p.name] for p in paths]]
E_txt = embed_texts(options.ravel().tolist()).reshape(len(mcq), 4, -1)

sims = np.einsum("nd,nkd->nk", E_img, E_txt)      # cosine: everything is unit-norm
pred = sims.argmax(1)
correct = pred == answer

acc = correct.mean()
print(f"\naccuracy = {acc:.4f}   (chance 0.25, n = {len(mcq)})")

In [ ]:
res = pd.DataFrame({
    "true_template": mcq["correct_answer_template"].values,
    "pred_type": opt_type[np.arange(len(mcq)), pred],
    "correct": correct,
    "margin": sims.max(1) - np.take_along_axis(sims, answer[:, None], 1).ravel(),
})

by_t = (res.groupby("true_template")["correct"]
           .agg(n="size", accuracy="mean")
           .reindex(["positive", "negative", "hybrid"]))
by_t["vs_chance"] = by_t["accuracy"] - 0.25
display(by_t.style.format({"accuracy": "{:.3f}", "vs_chance": "{:+.3f}"}))

gap = by_t.loc["positive", "accuracy"] - by_t.loc["negative", "accuracy"]
print(f"positive - negative gap = {gap:+.3f}")

fig, ax = plt.subplots(figsize=(5.4, 3.2))
ax.bar(by_t.index, by_t["accuracy"], color=["#4C72B0", "#C44E52", "#DD8452"], width=0.6)
ax.axhline(0.25, color="k", ls=":", lw=1)
ax.text(2.45, 0.26, "chance", fontsize=7, ha="right")
for i, v in enumerate(by_t["accuracy"]):
    ax.text(i, v + 0.012, f"{v:.3f}", ha="center", fontsize=8)
ax.set(ylabel="accuracy", ylim=(0, 1.0), title="MCQ accuracy by template of the correct answer")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); plt.show()

### The confusion matrix

Rows: the template of the **correct** caption. Columns: the type of the caption CLIP actually **chose**.
The diagonal is not accuracy — a right-type pick can still be the wrong caption — so the correct/incorrect
matrix is given separately underneath. What the off-diagonal shows is the *shape* of the error: a model
with an affirmation bias sends its errors into the `positive` column whatever the question was.


In [ ]:
order = ["positive", "negative", "hybrid"]
M = pd.crosstab(res["true_template"], res["pred_type"]).reindex(index=order, columns=order).fillna(0)
Mn = M.div(M.sum(axis=1), axis=0)

# binary version: was the answer right or wrong, per template
B = pd.crosstab(res["true_template"], np.where(res["correct"], "correct", "wrong")).reindex(order)
B = B.reindex(columns=["correct", "wrong"]).fillna(0).astype(int)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for a, (D, title, fmt) in zip(ax, [
        (Mn, "chosen caption TYPE (row-normalised)", "{:.2f}"),
        (B.div(B.sum(axis=1), axis=0), "correct vs wrong (row-normalised)", "{:.2f}")]):
    im = a.imshow(D.values, cmap="Blues", vmin=0, vmax=1)
    a.set_xticks(range(D.shape[1]), D.columns, fontsize=8)
    a.set_yticks(range(D.shape[0]), D.index, fontsize=8)
    a.set(xlabel="predicted", ylabel="true template", title=title)
    for i in range(D.shape[0]):
        for j in range(D.shape[1]):
            v = D.values[i, j]
            a.text(j, i, fmt.format(v), ha="center", va="center", fontsize=9,
                   color="white" if v > 0.55 else "black")
fig.tight_layout(); plt.show()

print("counts — chosen type:"); print(M.astype(int).to_string())
print("\ncounts — correct/wrong:"); print(B.to_string())
frac_pos = (res["pred_type"] == "positive").mean()
frac_pos_err = (res.loc[~res["correct"], "pred_type"] == "positive").mean()
print(f"\nCLIP picks a purely affirmative caption in {frac_pos:.1%} of questions "
      f"(a third of the options are; {frac_pos_err:.1%} of its ERRORS land there)")

## Retrieval-Neg

In [ ]:
ret = pd.read_csv(CONFIG["retrieval_csv"])
ret_paths = [resolve(p) for p in ret["filepath"]]
E_ret_img = embed_images(ret_paths)

queries, gold = [], []
for i, caps in enumerate(ret["captions"]):
    for c in ast.literal_eval(caps):
        queries.append(c); gold.append(i)
gold = np.asarray(gold)
print(f"{len(ret)} images | {len(queries)} negated captions")

E_q = embed_texts(queries)
S = E_q @ E_ret_img.T                       # (queries, images) cosine
kmax = max(CONFIG["recall_at"])
topk = np.argpartition(-S, kmax, axis=1)[:, :kmax]
topk = np.take_along_axis(topk, np.argsort(-np.take_along_axis(S, topk, 1), axis=1), axis=1)
hit_rank = np.where(topk == gold[:, None])

recalls = {f"R@{k}": float(np.mean([(topk[:, :k] == gold[:, None]).any(1).mean()]))
           for k in CONFIG["recall_at"]}
rank = np.full(len(queries), np.inf)
rank[hit_rank[0]] = hit_rank[1] + 1
for k, v in recalls.items():
    print(f"  {k} = {v:.4f}   (chance {k.split('@')[1]}/{len(ret)} = {int(k.split('@')[1]) / len(ret):.5f})")
print(f"  median rank of the correct image (when in top {kmax}) = "
      f"{np.median(rank[np.isfinite(rank)]):.0f}")

### Does the retrieved image actually contain the negated object?

The query says *"…but there is no fork"*. The file lists, per image, the objects that **are** present and
the ones that are **absent**. So the failure can be characterised precisely rather than counted: when CLIP
returns the wrong image, does that image contain the very object the caption said was missing? That is the
affirmation bias measured on the object level, not on a benchmark's option list.


In [ ]:
present = [set(ast.literal_eval(s)) for s in ret["positive_objects"]]
absent = [set(ast.literal_eval(s)) for s in ret["negative_objects"]]
vocab = sorted(set().union(*present, *absent))

top1 = topk[:, 0]
wrong = top1 != gold
# the object the caption negates: the gold image's absent objects mentioned in the query text
# P(object present) per object, over the whole image set -> the matched baseline
prevalence = {o: float(np.mean([o in p for p in present])) for o in vocab}

n_checked = n_contains = 0
base = []
for q, g, t in zip(np.flatnonzero(wrong), gold[wrong], top1[wrong]):
    negated = {o for o in absent[g] if re.search(rf"\b{re.escape(o)}s?\b", queries[q], re.I)}
    if not negated:
        continue
    n_checked += 1
    n_contains += bool(negated & present[t])
    # chance that a RANDOM image contains at least one of these objects, same objects
    base.append(1.0 - np.prod([1.0 - prevalence[o] for o in negated]))

print(f"wrong top-1: {wrong.sum()}/{len(queries)} ({wrong.mean():.1%})")
print(f"of the {n_checked} errors where the negated object could be identified in the query,")
print(f"  {n_contains} ({n_contains / max(n_checked, 1):.1%}) return an image that CONTAINS it")
print(f"  matched baseline (a RANDOM image, same negated objects): {np.mean(base):.1%}")
print(f"  ratio = {(n_contains / max(n_checked, 1)) / max(np.mean(base), 1e-9):.1f}x")

## Qualitative failures

In [ ]:
bad = res.index[(~res["correct"]) & (res["true_template"] == "negative")][:CONFIG["n_examples"]]
for i in bad:
    fig, ax = plt.subplots(1, 2, figsize=(9.5, 2.9), gridspec_kw={"width_ratios": [1, 1.9]})
    ax[0].imshow(Image.open(paths[i]).convert("RGB")); ax[0].axis("off")
    ax[0].set_title(paths[i].name, fontsize=7)
    o = np.argsort(-sims[i])
    lab = [f"{'✓' if j == answer[i] else ' '} {str(options[i][j])[:62]}" for j in o]
    ax[1].barh(np.arange(4)[::-1], sims[i][o],
               color=["#55A868" if j == answer[i] else "#C44E52" for j in o], height=0.6)
    ax[1].set_yticks(np.arange(4)[::-1], lab, fontsize=7)
    ax[1].set_xlim(sims[i].min() - 0.01, sims[i].max() + 0.02)
    ax[1].set_xlabel("cosine"); ax[1].spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); plt.show()